# Flame 与 CrewAI 集成示例

此可选示例让 CrewAI Agent 检索已建立索引的文档。请在独立环境中安装示例依赖，配置 `.env`，并先完成文档建库。


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Flame project directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import os
from crewai import Agent, Crew, LLM, Task
from crewai.tools import tool
from langchain_chroma import Chroma
from backend.embeddings import create_embeddings
from settings import CHROMA_PERSIST_DIRECTORY

vector_store = Chroma(
    persist_directory=CHROMA_PERSIST_DIRECTORY,
    embedding_function=create_embeddings(),
)
from citations import annotate_documents_for_citation, CITATION_MARKER_KEY


@tool("Search indexed documents")
def search_documents(query: str) -> str:
    """检索与完整用户问题相关的文档片段。"""
    documents, _ = annotate_documents_for_citation(
        vector_store.similarity_search(query, k=5)
    )
    return "\n\n".join(
        f"{doc.metadata[CITATION_MARKER_KEY]}\n{doc.page_content}".strip()
        for doc in documents
    )


In [ ]:
llm = LLM(
    model="openai/qwen3.8-27b",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
researcher = Agent(
    role="Document researcher",
    goal="Answer questions with evidence from indexed documents.",
    backstory="You find relevant passages and cite their supplied numbers.",
    tools=[search_documents],
    llm=llm,
    allow_delegation=False,
)
task = Task(
    description="Answer this question using only retrieved passages: {question}",
    expected_output="A concise answer with the supplied citation numbers, or a statement that the documents do not provide an answer.",
    agent=researcher,
)
crew = Crew(agents=[researcher], tasks=[task])


In [ ]:
result = crew.kickoff(inputs={"question": "What topics are covered by the indexed documents?"})
print(result.raw)
